In [2]:
import pandas as pd
from openai import OpenAI
import time
import re

# Data awal
data = [
    {
        "prompt": "Bagaimana cara mengatur ulang kata sandi saya di aplikasi e-commerce 'xx'?",
        "output": "1. Buka aplikasi\n2. Masuk ke halaman login\n3. Ketuk 'Lupa Kata Sandi'\n4. Masukkan email yang terdaftar\n5. Ketuk 'Kirim'\n6. Periksa email dan atur ulang kata sandi"
    },
    {
        "prompt": "Bagaimana cara menghapus akun saya di aplikasi 'xx'?",
        "output": "1. Buka aplikasi\n2. Masuk ke menu 'Pengaturan'\n3. Pilih 'Akun'\n4. Ketuk 'Hapus Akun'\n5. Masukkan PIN untuk konfirmasi"
    },
    {
        "prompt": "Bagaimana cara mengganti nomor telepon saya di aplikasi 'xx'?",
        "output": "1. Buka aplikasi\n2. Masuk ke menu 'Profil'\n3. Ketuk 'Edit Nomor Telepon'\n4. Masukkan nomor baru\n5. Verifikasi menggunakan OTP"
    },
    {
        "prompt": "Bagaimana cara memperbarui alamat pengiriman saya di aplikasi?",
        "output": "1. Buka aplikasi\n2. Masuk ke menu 'Pengaturan'\n3. Ketuk 'Alamat Pengiriman'\n4. Tambah atau edit alamat\n5. Simpan perubahan"
    },
    {
        "prompt": "Bagaimana cara mengaktifkan mode gelap di aplikasi 'xx'?",
        "output": "1. Buka aplikasi\n2. Masuk ke menu 'Pengaturan'\n3. Ketuk 'Tampilan'\n4. Pilih 'Mode Gelap'"
    }
]

df = pd.DataFrame(data)

# Setup OpenAI-compatible client
client = OpenAI(
    base_url="http://103.98.105.169:8080/v1",
    api_key="API KEY"  # API key
)

# Function untuk generate parafrase
def generate_variations(prompt, n=3):
    try:
        response = client.chat.completions.create(
            model="gemma-3-27b-it",
            temperature=0.8,
            top_p=0.95,
            max_tokens=512,
            messages=[
                {
                    "role": "system",
                    "content": "Kamu adalah asisten yang membantu membuat parafrase instruksi dalam Bahasa Indonesia. Selalu berikan tepat 3 variasi dalam format daftar bernomor yang bersih."
                },
                {
                    "role": "user",
                    "content": f"""Tolong parafrase instruksi berikut menjadi **tepat 3** versi yang singkat dan beragam.
Jawaban harus dalam bentuk daftar bernomor tanpa penjelasan tambahan. Hanya kalimat hasil parafrase saja.

Instruksi: "{prompt}" """
                }
            ]
        )
        raw_lines = response.choices[0].message.content.strip().split("\n")
        return [re.sub(r"^\d+\.\s*", "", line).strip() for line in raw_lines if line.strip()]
    
    except Exception as e:
        print(f"Terjadi error saat parafrase: {e}")
        return [prompt] * n


# Simpan hasil variasi
generated_data = []

for i, row in df.iterrows():
    variations = generate_variations(row["prompt"], n=3)
    for j, v in enumerate(variations):
        generated_data.append({
            "prompt": v.strip(" -123. "),  # Bersihkan bullet jika ada
            "output": row["output"],
            "variasi_label": f"intent_{i+1}_var_{j+1}"
        })
    time.sleep(1)  # Hindari flood API jika perlu

final_df = pd.DataFrame(generated_data)
print(final_df)

# Simpan ke CSV jika ingin
# final_df.to_csv("synthetic_variations.csv", index=False)


                                               prompt  \
0   Reset kata sandi aplikasi 'xx' bagaimana caranya?   
1                Panduan mengubah kata sandi di 'xx'?   
2   Saya lupa kata sandi 'xx', bagaimana mengaturn...   
3               Cara menghapus akun di aplikasi 'xx'?   
4                       Panduan penghapusan akun 'xx'   
5   Saya ingin menghapus akun aplikasi 'xx', bagai...   
6   Ganti nomor telepon di aplikasi 'xx' bagaimana...   
7   Saya ingin mengubah nomor telepon di aplikasi ...   
8    Langkah-langkah mengganti nomor telepon di 'xx'?   
9   Ubah alamat pengiriman di aplikasi, bagaimana ...   
10  Panduan memperbarui alamat pengiriman dalam ap...   
11  Saya ingin mengganti alamat pengiriman di apli...   
12           Aktifkan tampilan gelap di aplikasi 'xx'   
13   Cara mengganti tema aplikasi 'xx' ke mode gelap?   
14        Bagaimana agar aplikasi 'xx' menjadi gelap?   

                                               output   variasi_label  
0   1. Buka apl